In [1]:
import os

In [2]:
%pwd

'c:\\Users\\shane\\OneDrive\\Desktop\\AI Projects\\detection\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\shane\\OneDrive\\Desktop\\AI Projects\\detection'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from chicken_disease_detection.constants import *

from chicken_disease_detection.constants import *
from chicken_disease_detection.utils.common import read_yaml, create_directories

In [18]:
class ConfigurationManager:
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
        create_directories([self.config.data_ingestion.root_dir])
    

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        data_ingestion_config = DataIngestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )

        return data_ingestion_config

In [19]:
import os
import urllib.request as request
import zipfile
from chicken_disease_detection import logger
from chicken_disease_detection.utils.common import get_size


In [26]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL, 
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} downloaded with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [27]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    try:
        data_ingestion.download_file()
        logger.info(f"File downloaded! Size: {get_size(Path(data_ingestion_config.local_data_file))}")
        data_ingestion.extract_zip_file()
        logger.info("Data ingestion completed successfully!")
    except Exception as e:
        logger.error(f"Error downloading file: {e}")
except Exception as e:
    logger.error(f"Error occurred while initializing data ingestion: {e}")

[2026-05-28 13:59:53,504: INFO: common]: YAML file 'config\config.yaml' read successfully.
[2026-05-28 13:59:53,513: INFO: common]: YAML file 'params.yaml' read successfully.
[2026-05-28 13:59:53,518: INFO: common]: Directory created: artifacts
[2026-05-28 13:59:53,522: INFO: common]: Directory created: artifacts/data_ingestion


[2026-05-28 13:59:53,526: INFO: 3631545635]: File already exists of size: ~ 11.079 MB
[2026-05-28 13:59:53,527: INFO: 2553491662]: File downloaded! Size: ~ 11.079 MB
[2026-05-28 13:59:54,053: INFO: 2553491662]: Data ingestion completed successfully!
